# (If using Colab)Access our github repo

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!git clone https://github.com/rohankumar-1/multimodal-seizure-detection.git

Cloning into 'multimodal-seizure-detection'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 58 (delta 27), reused 41 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 21.79 KiB | 3.11 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [3]:
import os

os.chdir("./multimodal-seizure-detection")

In [4]:
# try to update to make sure we are up to date
!git fetch

# Load in data, and apply preprocessing as necessary

In [2]:
import numpy as np

# train_data = np.load("/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/train_data.npz", allow_pickle=True)
# val_data = np.load("/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/val_data.npz", allow_pickle=True)
# test_data = np.load("/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/test_data.npz", allow_pickle=True)

train_data = np.load("data/train_data_2sec.npz", allow_pickle=True)
val_data = np.load("data/val_data_2sec.npz", allow_pickle=True)
test_data = np.load("data/test_data_2sec.npz", allow_pickle=True)


In [3]:
train_eeg, train_ecg = train_data['eeg'], train_data['ecg']
val_eeg, val_ecg = val_data['eeg'], val_data['ecg']
test_eeg, test_ecg = test_data['eeg'], test_data['ecg']

In [83]:
# this function returns a preprocessed tensor for the given modality, with filtering + normalization
# Parameters:
#   desired_fs: frequency of returned signal
#   lowcut: lower cut of the bandwith filter
#   highcut: higher cut of the bandwith filter
# output is tuple of tensors
from src import preprocess_signal_nn

train_eeg, val_eeg, test_eeg = preprocess_signal_nn(train_eeg, val_eeg, test_eeg, desired_fs=8, lowcut=0.5, highcut=60.0, notch_freq=50.0)
train_ecg, val_ecg, test_ecg = preprocess_signal_nn(train_ecg, val_ecg, test_ecg, desired_fs=8, lowcut=0.5, highcut=60.0, notch_freq=50.0)

In [84]:
import torch
from src import SupervisedMultimodalDataset

trainset = SupervisedMultimodalDataset({'eeg': train_eeg, 'ecg':train_ecg}, torch.tensor(train_data['binary_label']))
valset = SupervisedMultimodalDataset({'eeg': val_eeg, 'ecg': val_ecg}, torch.tensor(val_data['binary_label']))
testset = SupervisedMultimodalDataset({'eeg': test_eeg, 'ecg': test_ecg}, torch.tensor(test_data['binary_label']))

In [85]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=8, shuffle=True)
valloader = DataLoader(valset, batch_size=8, shuffle=False)
testloader = DataLoader(testset, batch_size=8, shuffle=False)

# SVM

In [87]:
# SVM Model
from src.models import SVMModel


train_eeg_flat = train_eeg.view(train_eeg.size(0), -1)
test_eeg_flat = test_eeg.view(test_eeg.size(0), -1)


m = SVMModel(kernel='rbf')

In [92]:
import torch

N_samples = 5000
indices = torch.randperm(train_eeg_flat.size(0))[:N_samples]

train_eeg_flat_subsample = train_eeg_flat[indices]

In [93]:
m.fit(train_eeg_flat_subsample, train_data['binary_label'][indices])

In [94]:
results = m.evaluate(test_eeg_flat, test_data['binary_label'], verbose=False)

In [95]:
results

{'auc': 0.7271707178982967,
 'accuracy': 0.8004451510333863,
 'f1': 0.7146617277156844,
 'precision': 0.7552145226551963,
 'recall': 0.8004451510333863,
 'fpr': array([0.00000000e+00, 0.00000000e+00, 2.38473768e-04, ...,
        9.99364070e-01, 9.99364070e-01, 1.00000000e+00]),
 'tpr': array([0.00000000e+00, 3.17965024e-04, 3.17965024e-04, ...,
        9.99364070e-01, 1.00000000e+00, 1.00000000e+00]),
 'thresholds': array([       inf, 0.90757387, 0.85478295, ..., 0.16052268, 0.15917059,
        0.14913793])}

# Basic Fusion

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TwoModalityMLP(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        # EEG: (B, 2, 2560) → flatten → (B, 5120)
        self.eeg_lin = nn.Linear(512, hidden)

        # ECG: (B, 1, 2560)
        self.ecg_conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)   # → (B, 32, 1)
        )
        self.ecg_lin = nn.Linear(32, hidden)

        # Fusion MLP
        self.fuse = nn.Sequential(
            nn.ReLU(),
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, **kwargs):
        eeg = kwargs["eeg"]   # (B, 2, 2560)
        ecg = kwargs["ecg"]   # (B, 1, 2560)

        # EEG flatten
        eeg_flat = eeg.reshape(eeg.size(0), -1)  # (B, 5120)
        h1 = F.relu(self.eeg_lin(eeg_flat))

        # ECG conv
        ecg_feat = self.ecg_conv(ecg).squeeze(-1)  # (B, 32)
        h2 = F.relu(self.ecg_lin(ecg_feat))

        # Fusion
        h = torch.cat([h1, h2], dim=1)
        return self.fuse(h)

In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
len(valset)

44425

In [64]:
from src import train_supervised_nn

model = TwoModalityMLP()
train_supervised_nn(model, trainloader, valloader, task='classification', pos_weight=4.0, lr=1e-4, checkpoint_path='weights/mlp_2.pth', patience=2, device='cpu', epochs=10)

Epoch 1:   0%|          | 0/6130 [00:00<?, ?it/s]

Epoch 1: 100%|██████████| 6130/6130 [00:05<00:00, 1038.12it/s]


Epoch 1 | Val AUC: 0.8120
Saved new best model at epoch 1 (loss=0.8788)


Epoch 2: 100%|██████████| 6130/6130 [00:05<00:00, 1149.08it/s]


Epoch 2 | Val AUC: 0.8089
Saved new best model at epoch 2 (loss=0.8772)


...

Epoch 3 | Val AUC: 0.8135
Saved new best model at epoch 3 (loss=0.8689)


Epoch 4: 100%|██████████| 6130/6130 [00:05<00:00, 1114.18it/s]


Epoch 4 | Val AUC: 0.8140


Epoch 5: 100%|██████████| 6130/6130 [00:05<00:00, 1124.25it/s]


Epoch 5 | Val AUC: 0.8080
Early stopping triggered at epoch 5


In [81]:
from src import evaluate_nn, find_best_threshold

thresh = find_best_threshold(model, testloader, metric='f1', device='cpu')
results = evaluate_nn(model, testloader, thresh, 'cpu')

results

{'auc_score': 0.8065651436529582,
 'accuracy': 0.8032432432432433,
 'f1': 0.5394462637689789,
 'precision': 0.5071368597816961,
 'recall': 0.5761526232114468,
 'fpr': array([0.00000000e+00, 7.94912560e-05, 7.94912560e-05, ...,
        9.99841017e-01, 9.99841017e-01, 1.00000000e+00]),
 'tpr': array([0.00000000e+00, 6.35930048e-04, 1.27186010e-03, ...,
        9.99682035e-01, 1.00000000e+00, 1.00000000e+00]),
 'thresholds': array([          inf, 9.9999988e-01, 9.9999976e-01, ..., 6.6770815e-05,
        5.9112976e-05, 1.2113493e-06], dtype=float32)}

# ChronoNet (EEG)

In [7]:
from src.models import ChronoNet

class ChronoNetConfig:
  fs = 256
  frame = 10
  CH = 2
  dropoutRate = 0.2
  l2 = 0.0001
  cnn_drop = 0.35
  batchnorm = True
  maxpool = False
  avgpool = False
  strided = False

cn = ChronoNet(ChronoNetConfig())

In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
from src import train_supervised_nn

train_supervised_nn(cn, trainloader, valloader, task='classification', pos_weight=4.0, lr=1e-4, device='cpu', epochs=10)

NameError: name 'cn' is not defined

# MatrixProfile

In [96]:
from src.models import MatrixProfile

In [ ]:
from src import apply_bandpass, apply_notch

from scipy.signal import resample

# example usage
data = np.load("data/testruns/testruns/testrun_0.npz", allow_pickle=True)
ecg = data['ecg']
labels = data['binary_label']

ecg = ecg.squeeze()

ecg = apply_bandpass(ecg, 0.5, 60, 256)
ecg = apply_notch(ecg, 50, 256, 35)

test_len = int(ecg.shape[-1] * 8 / 256.0)
ecg = resample(ecg, test_len, axis=-1)
ecg_flatten = ecg.flatten()

In [105]:
ecg.shape

(80, 2560)

In [101]:
ecg_flatten.shape

(204800,)

In [102]:
mp = MatrixProfile(m=2560, percentile=98, min_sep=10, min_cluster=3, max_gap=5)

out = mp.predict(ecg_flatten)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
Exception ignored on calling ctypes callback function: <function ExecutionEngine._raw_object_cache_notify at 0x17a222dc0>
Traceback (most recent call last):
  File "/Users/roku/miniconda3/envs/ml-torch/lib/python3.9/site-packages/llvmlite/binding/executionengine.py", line 178, in _raw_object_cache_notify
    def _raw_object_cache_notify(self, data):
KeyboardInterrupt: 


In [ ]:
out['mp']